In [1]:
# CAPSTONE PROJECT - DATA SCIENCE PGC
# Script 1: Web Scraping - Books to Scrape
# Website: https://books.toscrape.com
# ============================================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://books.toscrape.com/catalogue/"
START_URL = "https://books.toscrape.com/catalogue/page-1.html"

RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

books = []

def scrape_book_detail(book_url):
    """Scrape individual book page for extra details."""
    try:
        res = requests.get(book_url, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        # Genre / Category
        breadcrumb = soup.select("ul.breadcrumb li")
        genre = breadcrumb[-2].text.strip() if len(breadcrumb) >= 3 else "Unknown"

        # Table data
        table = soup.find("table", class_="table-striped")
        table_data = {}
        if table:
            for row in table.find_all("tr"):
                header = row.find("th").text.strip()
                value = row.find("td").text.strip()
                table_data[header] = value

        availability = table_data.get("Availability", "Unknown")
        num_in_stock = ''.join(filter(str.isdigit, availability))
        num_in_stock = int(num_in_stock) if num_in_stock else 0
        num_reviews = int(table_data.get("Number of reviews", 0))

        # Description
        desc_tag = soup.find("div", id="product_description")
        description = desc_tag.find_next_sibling("p").text.strip() if desc_tag else ""

        return genre, num_in_stock, num_reviews, description
    except Exception as e:
        print(f"  [Detail Error] {e}")
        return "Unknown", 0, 0, ""


def scrape_all_books():
    url = START_URL
    page = 1

    while url:
        print(f"Scraping page {page}...")
        try:
            res = requests.get(url, timeout=10)
            soup = BeautifulSoup(res.text, "html.parser")
        except Exception as e:
            print(f"  [Page Error] {e}")
            break

        book_cards = soup.select("article.product_pod")

        for card in book_cards:
            # Title
            title = card.h3.a["title"]

            # Price
            price_str = card.select_one("p.price_color").text.strip()
            price = float(price_str.replace("£", "").replace("Â", "").strip())

            # Rating
            rating_word = card.p["class"][1]
            rating = RATING_MAP.get(rating_word, 0)

            # Availability
            avail = card.select_one("p.availability").text.strip()

            # Book detail URL
            relative_url = card.h3.a["href"].replace("../", "")
            book_url = BASE_URL + relative_url

            # Get detail page info
            genre, num_in_stock, num_reviews, description = scrape_book_detail(book_url)

            books.append({
                "title": title,
                "price_gbp": price,
                "rating": rating,
                "rating_label": rating_word,
                "availability": avail,
                "num_in_stock": num_in_stock,
                "num_reviews": num_reviews,
                "genre": genre,
                "description": description,
                "url": book_url
            })

            time.sleep(0.05)  # polite delay

        # Find next page
        next_btn = soup.select_one("li.next a")
        if next_btn:
            next_href = next_btn["href"]
            url = BASE_URL + next_href
            page += 1
        else:
            url = None

    print(f"\n✅ Total books scraped: {len(books)}")
    return books


if __name__ == "__main__":
    data = scrape_all_books()
    df = pd.DataFrame(data)
    df.to_csv("books_raw.csv", index=False)
    print("✅ Data saved to books_raw.csv")
    print(df.head())
    print(f"\nShape: {df.shape}")
    print(df.dtypes)


Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page 45...
Scraping page 46...
Scraping page 47...
Scraping page 48...
Scraping page 49...
Scraping page 50...

✅ Total 